# Restaurant search — GPU playground (Approaches 6 & 7)

Evaluate and experiment with the GPU-heavy search approaches against the
209-query benchmark (NDCG@5 / Recall@10 / MRR):

- **Approach 6** — cross-encoder scoring every menu item (`crossencoder`)
- **Approach 7** — local LLM as reader via vLLM (`llm_search.py`)
- plus the cheap baselines (`text`, `enhanced`) and optionally ColBERT / SBERT for comparison

**Before running:** *Runtime → Change runtime type → GPU* (L4 or A100 recommended;
a T4 works with a ≤3B model, see the Approach 7 cell).
If the repo is private, add a Colab secret named **`GHTOKEN`** (key icon in the left sidebar)
holding a GitHub token with read access to `ahsank/restaurant-search`.

Run the install cell **first**, before anything imports `torch`.

In [ ]:
# --- Environment detection + code checkout -----------------------------------
import os, subprocess, sys

def detect_environment():
    if 'google.colab' in sys.modules or 'COLAB_RELEASE_TAG' in os.environ:
        return 'Colab'
    if os.environ.get('KAGGLE_KERNEL_RUN_TYPE'):
        return 'Kaggle'
    return 'Local/Other'

ENV = detect_environment()
print(f'Current environment: {ENV}')

REPO = 'ahsank/restaurant-search'
BRANCH = 'claude/approach7-llm-reader'   # falls back to main if this branch is gone

if ENV == 'Colab':
    from google.colab import userdata
    try:
        ghtoken = userdata.get('GHTOKEN')
    except Exception:          # secret not defined / not shared with this notebook
        ghtoken = ''
    root_dir = '/content'
elif ENV == 'Kaggle':
    from kaggle_secrets import UserSecretsClient
    try:
        ghtoken = UserSecretsClient().get_secret('gh-token')
    except Exception:
        ghtoken = ''
    root_dir = '/kaggle/working'
else:
    ghtoken = ''
    root_dir = None            # running from a local checkout: research/colab/ -> repo root

if root_dir is not None:
    REPO_DIR = f'{root_dir}/restaurant-search'
    auth = f'ahsank:{ghtoken}@' if ghtoken else ''
    url = f'https://{auth}github.com/{REPO}.git'
    subprocess.run(['rm', '-rf', REPO_DIR], check=True)
    for branch in (BRANCH, 'main'):
        r = subprocess.run(['git', 'clone', '-q', '--depth', '1', '--branch', branch, url, REPO_DIR],
                           capture_output=True, text=True)
        if r.returncode == 0:
            print(f'Cloned {REPO}@{branch} -> {REPO_DIR}')
            break
        print(f'Clone of {branch} failed: {r.stderr.replace(ghtoken, "***") if ghtoken else r.stderr}')
    else:
        raise RuntimeError('Could not clone the repo — set the GHTOKEN secret if it is private.')
else:
    REPO_DIR = os.path.abspath('../..')

for p in (f'{REPO_DIR}/research', f'{REPO_DIR}/research/search'):
    if p not in sys.path:
        sys.path.insert(0, p)
os.chdir(REPO_DIR)

In [ ]:
# --- Install GPU dependencies (run before anything imports torch) -------------
# vLLM pins its own torch; sentence-transformers / pylate then reuse that torch.
# Colab prints dependency-conflict warnings for packages we don't use (RAPIDS, ADK) — harmless.
INSTALL_VLLM = True      # Approach 7
INSTALL_PYLATE = False   # ColBERT (optional comparison)

pkgs = ['sentence-transformers'] + (['vllm'] if INSTALL_VLLM else []) + (['pylate'] if INSTALL_PYLATE else [])
%pip install -q {' '.join(pkgs)}
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv

In [ ]:
# --- Shared setup: data, benchmark, engine helper ---------------------------
import json, time
from pathlib import Path
import pandas as pd

import engines
from evaluate import evaluate, load_queries
from lib.data import load_data, judged_restaurant_ids

restaurants, menus = load_data(judged_restaurant_ids())
QUERIES = load_queries()
print(f'{len(restaurants)} restaurants, {sum(map(len, menus.values()))} items, {len(QUERIES)} queries')

RESULTS = {}            # engine label -> evaluate() output
SEARCH_FNS = {}         # engine label -> search_fn(query) for the playground cells

def build_engine(engine, model=None, **flags):
    # engines.build_arg_parser pre-reads sys.argv to pick engine-specific flags;
    # point it at our args instead of the kernel's.
    argv = ['--engine', engine] + (['--model', model] if model else [])
    saved, sys.argv = sys.argv, ['notebook', *argv]
    try:
        args = engines.build_arg_parser('notebook').parse_args(argv)
    finally:
        sys.argv = saved
    for k, v in flags.items():
        setattr(args, k, v)
    return engines.build_search_fn(args, restaurants, menus)

def score(label, search_fn):
    t0 = time.time()
    RESULTS[label] = evaluate(search_fn)
    SEARCH_FNS[label] = search_fn
    o = RESULTS[label]['overall']
    print(f"{label}: NDCG@5={o['ndcg@5']:.4f}  Recall@10={o['recall@10']:.4f}  MRR={o['mrr']:.4f}"
          f"  ({time.time() - t0:.0f}s)")

def summary():
    rows = {k: {m: v['overall'][m] for m in ('ndcg@5', 'recall@10', 'mrr')} for k, v in RESULTS.items()}
    return pd.DataFrame(rows).T.sort_values('ndcg@5', ascending=False).round(4)

def by(key):   # key: 'by_category' or 'by_difficulty'
    return pd.DataFrame({k: {c: m['ndcg@5'] for c, m in v[key].items()} for k, v in RESULTS.items()}).round(3)

## Baselines (CPU, seconds)

In [ ]:
score('text', build_engine('text'))
score('enhanced', build_engine('enhanced'))

## Approach 6 — cross-encoder (GPU)

Scores every one of the 795 items per query. ~4 s/query on CPU; a few minutes for the
whole benchmark on a GPU. Swap `CE_MODEL` for any `sentence-transformers` CrossEncoder
checkpoint (e.g. `cross-encoder/ms-marco-MiniLM-L12-v2`, `BAAI/bge-reranker-base`).

In [ ]:
CE_MODEL = 'cross-encoder/ms-marco-MiniLM-L6-v2'
score(f'crossencoder:{CE_MODEL.split("/")[-1]}', build_engine('crossencoder', CE_MODEL))

Optional — ColBERT / SBERT for comparison (set `INSTALL_PYLATE = True` above for ColBERT).

In [ ]:
RUN_EXTRA = False
if RUN_EXTRA:
    score('sbert:all-mpnet-base-v2', build_engine('sbert', 'all-mpnet-base-v2'))
    if INSTALL_PYLATE:
        score('colbert', build_engine('colbert'))

## Approach 7 — local LLM as reader (vLLM, GPU)

One prompt per (query, restaurant): the restaurant's whole menu as a numbered list, the query
last (so vLLM's prefix cache reuses each menu across queries). The model answers with graded
`[item_number, grade]` pairs; restaurants are merged by grade, then rank.

| GPU | Suggested `LLM_MODEL` |
|---|---|
| T4 (16 GB) | `Qwen/Qwen2.5-3B-Instruct`, `microsoft/Phi-3.5-mini-instruct` |
| L4 (24 GB) | `Qwen/Qwen2.5-7B-Instruct` (default) |
| A100 (40/80 GB) | `Qwen/Qwen2.5-14B-Instruct`, `Qwen/Qwen2.5-32B-Instruct-AWQ` |

Gated models (Llama) need an `HF_TOKEN` Colab secret. To try another model, restart the
runtime (vLLM doesn't release GPU memory cleanly) and re-run the setup cells.

In [ ]:
import llm_search

LLM_MODEL = 'Qwen/Qwen2.5-7B-Instruct'
GPU_MEM = 0.85    # leave room for the cross-encoder loaded above

if ENV == 'Colab':
    try:
        os.environ.setdefault('HF_TOKEN', userdata.get('HF_TOKEN'))
    except Exception:
        pass

import torch; torch.cuda.empty_cache()
llm = llm_search.load_llm(LLM_MODEL, gpu_memory_utilization=GPU_MEM)

In [ ]:
# Full benchmark: 209 queries x 8 restaurants = 1,672 prompts (a few minutes on an L4).
# Rankings JSONL can be replayed anywhere: evaluate.py --engine llm --rankings FILE
tag = LLM_MODEL.split('/')[-1].lower()
RANKINGS = Path(f'research/results/llm/{tag}.jsonl')
llm_search.run(QUERIES, restaurants, menus, LLM_MODEL, RANKINGS, llm=llm)
score(f'llm:{tag}', llm_search.replay_search_fn(RANKINGS))

## Compare

In [ ]:
display(summary())
display(by('by_difficulty'))
by('by_category')

## Playground — try your own queries

`SEARCH_FNS` holds every engine scored above; the LLM entry replays benchmark rankings,
so the live LLM is wired in separately.

In [ ]:
def show(query, k=8):
    fns = dict(SEARCH_FNS)
    if 'llm' in globals():
        fns[f'llm:{LLM_MODEL.split("/")[-1].lower()} (live)'] = lambda q: llm_search.search(q, llm, restaurants, menus)
    cols = {}
    for label, fn in fns.items():
        if label.startswith('llm:') and '(live)' not in label:
            continue
        hits = fn(query)[:k]
        cols[label] = [f"{h['name']} · {restaurants[h['restaurant_id']]['name']}"
                       + (f" [{h['grade']}]" if 'grade' in h else '') for h in hits]
        cols[label] += [''] * (k - len(cols[label]))
    return pd.DataFrame(cols, index=range(1, k + 1))

show('seafood italian')

In [ ]:
show('something light and vegetarian for lunch')

## Where does the LLM miss? (per-query NDCG@5, worst first)

In [ ]:
llm_key = next((k for k in RESULTS if k.startswith('llm:')), None)
other = next((k for k in RESULTS if k.startswith('crossencoder')), None)
if llm_key:
    pq = pd.DataFrame(RESULTS[llm_key]['per_query']).set_index('query_id')
    if other:
        pq[other] = pd.DataFrame(RESULTS[other]['per_query']).set_index('query_id')['ndcg@5']
    display(pq[pq.n_relevant > 0].sort_values('ndcg@5')
              [['query_text', 'category_name', 'difficulty', 'ndcg@5'] + ([other] if other else [])].head(20))

In [ ]:
# Inspect the raw LLM answer for one query (per restaurant)
QID = 'C03-01'
row = next(json.loads(l) for l in open(RANKINGS) if json.loads(l).get('query_id') == QID)
print(row['query_text'])
for rid, text in row['raw'].items():
    print(f'  {rid:<24} {text[:200]}')

## Save results

Download `research/results/` (or copy it to Drive) to keep rankings and metrics:
commit the `.jsonl` files so others can replay them with `evaluate.py --engine llm --rankings`.

In [ ]:
summary().to_csv('research/results/summary.csv')
if ENV == 'Colab':
    !cd research && zip -qr /content/results.zip results
    from google.colab import files
    files.download('/content/results.zip')